# Notebook 02 — Create the Labels

## Objective

The goal of this notebook is to create the target variable for the late-delivery prediction problem.

An order is labeled as **late** when the actual customer delivery date is later than the estimated delivery date.

This notebook reads the joined order-level artifact produced by Notebook 1 and saves a labeled order-level dataset for the train/validation/test split stage.

### Input Artifact

`artifacts/01_joined/ml_orders.parquet`

### Output Artifact

`artifacts/02_labeled/labeled_orders.parquet`

In [2]:
from pathlib import Path

import pandas as pd

In [3]:
PROJECT_ROOT = Path.cwd().parent

INPUT_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "01_joined"
    / "ml_orders.parquet"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "02_labeled"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "labeled_orders.parquet"

In [4]:
ml_table = pd.read_parquet(INPUT_PATH)

print("Shape:", ml_table.shape)
display(ml_table.head())

Shape: (99441, 37)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,unique_category_count,unique_seller_count_y,unique_seller_state_count,mean_product_weight_g,max_product_weight_g,mean_product_length_cm,mean_product_height_cm,mean_product_width_cm,mean_seller_latitude,mean_seller_longitude
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,1.0,1.0,1.0,500.0,500.0,19.0,8.0,13.0,-23.680729,-46.444238
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,1.0,1.0,1.0,400.0,400.0,19.0,13.0,19.0,-19.807681,-43.980427
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,1.0,1.0,1.0,420.0,420.0,24.0,19.0,21.0,-21.363502,-48.229601
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,1.0,1.0,1.0,450.0,450.0,30.0,10.0,20.0,-19.837682,-43.924053
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,1.0,1.0,1.0,250.0,250.0,51.0,15.0,15.0,-23.543395,-46.262086


## 2. Inspect the Delivery Columns

The target depends on comparing the actual customer delivery date with the estimated delivery date.



In [5]:
delivery_columns = [
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

ml_table[delivery_columns].info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   order_status                   99441 non-null  str           
 2   order_purchase_timestamp       99441 non-null  datetime64[us]
 3   order_delivered_customer_date  96476 non-null  datetime64[us]
 4   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](3), str(2)
memory usage: 7.7 MB


In [6]:
ml_table[delivery_columns].isna().sum()

order_id                            0
order_status                        0
order_purchase_timestamp            0
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [7]:
date_columns = [
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in date_columns:
    ml_table[col] = pd.to_datetime(
        ml_table[col],
        errors="coerce"
    )

In [8]:
missing_delivery = ml_table[
    ml_table["order_delivered_customer_date"].isna()
]

missing_delivery["order_status"].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [9]:
delivered_missing_date = ml_table[
    (ml_table["order_status"] == "delivered")
    & (ml_table["order_delivered_customer_date"].isna())
]

display(
    delivered_missing_date[
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
        ]
    ]
)

,order_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,NaT,2017-12-18
20618,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,NaT,2018-07-16
43834,2ebdfc4f15f23b91474edf87475f108e,delivered,2018-07-01 17:05:11,NaT,2018-07-30
79263,e69f75a717d64fc5ecdfae42b2e8e086,delivered,2018-07-01 22:05:55,NaT,2018-07-30
82868,0d3268bad9b086af767785e3f0fc0133,delivered,2018-07-01 21:14:02,NaT,2018-07-24
92643,2d858f451373b04fb5c984a1cc2defaf,delivered,2017-05-25 23:22:43,NaT,2017-06-23
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,2018-06-08 12:09:39,NaT,2018-06-26
98038,20edc82cf5400ce95e1afacc25798b31,delivered,2018-06-27 16:09:12,NaT,2018-07-19


### Missing Delivery Date Investigation

There are 2,965 orders with a missing `order_delivered_customer_date`.

Most of these records have non-delivered statuses such as `shipped`, `canceled`, `unavailable`, `invoiced`, or `processing`. Therefore, a missing delivery date should not be interpreted as a late delivery.

Eight orders are marked as `delivered` but still have a missing actual delivery date. Since the target requires comparing the actual delivery date with the estimated delivery date, these records cannot be labeled reliably.

For the late-vs-on-time prediction problem, only orders with a known delivery outcome and the dates required to calculate the target will be retained. Orders without sufficient information will be excluded rather than assigned an artificial label.

In [10]:
labeled_table = ml_table[
    (ml_table["order_status"] == "delivered")
    & ml_table["order_delivered_customer_date"].notna()
    & ml_table["order_estimated_delivery_date"].notna()
].copy()

print(f"All orders: {len(ml_table):,}")
print(f"Orders retained for labeling: {len(labeled_table):,}")
print(f"Orders excluded: {len(ml_table) - len(labeled_table):,}")

All orders: 99,441
Orders retained for labeling: 96,470
Orders excluded: 2,971


In [11]:
labeled_table["is_late"] = (
    labeled_table["order_delivered_customer_date"]
    > labeled_table["order_estimated_delivery_date"]
).astype("int8")

In [12]:
# is_late = 0  → On Time
# is_late = 1  → Late

In [13]:
labeled_table["delivery_label"] = (
    labeled_table["is_late"]
    .map({
        0: "On Time",
        1: "Late"
    })
)

In [14]:
validation_sample = labeled_table[
    [
        "order_id",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "is_late",
        "delivery_label",
    ]
].sample(10, random_state=42)

display(validation_sample)

,order_id,order_delivered_customer_date,order_estimated_delivery_date,is_late,delivery_label
9504,c6a73b421eb3e92ce86dbfbbd3530a8f,2018-03-13 17:13:17,2018-03-19,0,On Time
31123,b132124ca9d69faf63989e08a5851151,2018-06-04 19:26:52,2018-06-08,0,On Time
27721,7e25a1c58e68fa94300358caad65b944,2017-09-08 17:21:38,2017-09-25,0,On Time
74022,8418eb39cd68b52032566797b8f1bd11,2017-12-12 21:13:48,2017-12-20,0,On Time
36187,fe1ec86f91f3b5b6bc46fc3e4b8262cd,2017-12-13 01:16:52,2017-12-15,0,On Time
74560,c471a28c25283860362c67c0a1cd1694,2018-01-27 01:08:52,2018-02-22,0,On Time
27653,80e3ea9d65b89f8cbaa3d1f08f5dbd17,2018-06-19 21:50:42,2018-07-26,0,On Time
8691,a68c330c22c204ef5283a562bed75b97,2018-03-05 18:18:56,2018-03-13,0,On Time
50743,2728c4c5805dc2b5f0d4aee36cfde5d1,2017-06-19 18:16:09,2017-06-21,0,On Time
6938,d684202253f5f00621b0afd578646f3d,2017-10-10 20:21:31,2017-10-25,0,On Time


In [15]:
display(
    labeled_table[
        labeled_table["is_late"] == 1
    ][
        [
            "order_id",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "delivery_label",
        ]
    ].head(5)
)

,order_id,order_delivered_customer_date,order_estimated_delivery_date,delivery_label
20,203096f03d82e0dffbc41ebc2e2bcfb7,2017-10-09 22:23:46,2017-09-28,Late
25,fbf9ac61453ac646ce8ad9783d7d0af6,2018-03-21 22:03:54,2018-03-12,Late
35,8563039e855156e48fccee4d611a3196,2018-03-20 00:59:25,2018-03-20,Late
41,6ea2f835b4556291ffdc53fa0b3b95e8,2017-12-28 18:59:23,2017-12-21,Late
57,66e4624ae69e7dc89bd50222b59f581f,2018-04-03 13:28:46,2018-04-02,Late


## Class Distribution:

In [17]:
label_counts = (
    labeled_table["delivery_label"]
    .value_counts()
)

label_percent = (
    labeled_table["delivery_label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

label_summary = pd.DataFrame({
    "count": label_counts,
    "percentage": label_percent
})

label_summary

,count,percentage
delivery_label,,
On Time,88644,91.89
Late,7826,8.11


In [18]:
late_rate = labeled_table["is_late"].mean()

print(f"Late delivery rate: {late_rate:.2%}")
print(f"On-time delivery rate: {1 - late_rate:.2%}")

Late delivery rate: 8.11%
On-time delivery rate: 91.89%


## Class Imbalance Assessment

The class distribution is not perfectly balanced because late deliveries represent a smaller proportion of orders than on-time deliveries.

This means accuracy alone may not be sufficient for model evaluation. Later notebooks should also consider metrics such as precision, recall, F1-score, and ROC-AUC.

No resampling is performed at this stage. Any imbalance-handling strategy will be considered later using the training data only.

In [19]:
assert labeled_table["order_id"].is_unique
assert labeled_table["is_late"].isna().sum() == 0
assert set(labeled_table["is_late"].unique()).issubset({0, 1})

print("✓ One row per order preserved")
print("✓ No missing labels")
print("✓ Labels are binary")

✓ One row per order preserved
✓ No missing labels
✓ Labels are binary


In [21]:
print("Project root:", PROJECT_ROOT)
print("Output path:", OUTPUT_PATH)
print("Output directory exists:", OUTPUT_PATH.parent.exists())
print("Artifact exists:", OUTPUT_PATH.exists())

Project root: C:\Users\CW\Documents\Projects\Dataset_Olist
Output path: C:\Users\CW\Documents\Projects\Dataset_Olist\artifacts\02_labeled\labeled_orders.parquet
Output directory exists: True
Artifact exists: False


In [22]:
labeled_table.to_parquet(
    OUTPUT_PATH,
    index=False
)

print("Artifact saved:", OUTPUT_PATH)
print("Artifact exists:", OUTPUT_PATH.exists())

Artifact saved: C:\Users\CW\Documents\Projects\Dataset_Olist\artifacts\02_labeled\labeled_orders.parquet
Artifact exists: True


In [23]:
saved_labeled_table = pd.read_parquet(OUTPUT_PATH)

print("Reloaded shape:", saved_labeled_table.shape)
print("Unique order IDs:", saved_labeled_table["order_id"].is_unique)
print("Missing labels:", saved_labeled_table["is_late"].isna().sum())

Reloaded shape: (96470, 39)
Unique order IDs: True
Missing labels: 0
